# AI Dungeon — projeto Colab (End-to-End · 1 Clique)

Abra este arquivo no Colab, escolha uma GPU (T4 ou melhor) em Ambiente de execução → Alterar o tipo de ambiente de execução, e clique em **Executar tudo** (Run all). Nenhum segredo ou variável manual é exigido.

O notebook clona `https://github.com/Caiolinooo/aidungeon-local.git` em `main`, compila o llama-server (1648d6411) e baixa: qwen2.5-3b-instruct-q4_k_m.gguf, flux-2-klein-4b-fp8.safetensors, qwen_3_4b.safetensors, flux2-vae.safetensors.

No fim da execução ele imprime um link público `https://...trycloudflare.com/?aidl_token=...` que já entra logado em 1 clique (além do usuário `Caio` e senha gerada na hora).


In [ ]:
import subprocess
import sys
from pathlib import Path
REPO = Path('/content/aidungeon-local')
GIT_URL = 'https://github.com/Caiolinooo/aidungeon-local.git'
GIT_REF = 'main'
if not (REPO / ".git").is_dir():
    subprocess.check_call(["git", "clone", GIT_URL, str(REPO)])
subprocess.check_call(["git", "fetch", "origin", GIT_REF], cwd=REPO)
subprocess.check_call(["git", "checkout", "-f", GIT_REF], cwd=REPO)
subprocess.check_call(["git", "reset", "--hard", f"origin/{GIT_REF}"], cwd=REPO)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "backend/requirements.txt"], cwd=REPO)
print("repositório", GIT_REF)


In [ ]:
import os
import subprocess
import tarfile
import urllib.request
from pathlib import Path
REPO = Path('/content/aidungeon-local')
NODE_URL = 'https://nodejs.org/dist/v20.18.1/node-v20.18.1-linux-x64.tar.xz'
archive = Path("/tmp/node-colab.tar.xz")
if not list(Path("/usr/local/lib").glob("node-v*-linux-x64")):
    urllib.request.urlretrieve(NODE_URL, archive)
    with tarfile.open(archive) as bundle:
        bundle.extractall("/usr/local/lib")
extracted = next(Path("/usr/local/lib").glob("node-v*-linux-x64"))
for name in ("node", "npm", "npx"):
    dest = Path("/usr/local/bin") / name
    if dest.exists() or dest.is_symlink():
        dest.unlink()
    dest.symlink_to(extracted / "bin" / name)
env = os.environ.copy()
env["PATH"] = str(extracted / "bin") + ":/usr/local/bin:" + env.get("PATH", "")
subprocess.check_call(["npm", "ci"], cwd=REPO / "frontend", env=env)
subprocess.check_call(["npm", "run", "build"], cwd=REPO / "frontend", env=env)
print("frontend", REPO / "frontend" / "dist")


In [ ]:
import subprocess
import sys
from pathlib import Path
REPO = Path('/content/aidungeon-local')
COMFY_GIT = 'https://github.com/comfyanonymous/ComfyUI.git'
COMFY_REF = 'v0.46.0'
src = REPO / "ComfyUI"
if not (src / ".git").is_dir():
    subprocess.check_call(["git", "clone", COMFY_GIT, str(src)])
subprocess.check_call(["git", "fetch", "origin", "tag", COMFY_REF, "--no-tags"], cwd=src)
checkout = subprocess.run(["git", "checkout", COMFY_REF], cwd=src)
if checkout.returncode != 0:
    raise SystemExit("git checkout do ComfyUI falhou em " + COMFY_REF)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=src)
import torch
if not torch.cuda.is_available():
    raise SystemExit("PyTorch ficou sem CUDA depois do ComfyUI. Troque o ambiente para GPU e execute de novo.")
(REPO / "colab").mkdir(parents=True, exist_ok=True)
log = (REPO / "colab" / "comfyui.log").open("ab")
subprocess.run(["pkill", "-f", "main.py.*--port 8188"], check=False)
subprocess.Popen([
    sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188",
], cwd=src, stdout=log, stderr=log, start_new_session=True)
print("ComfyUI", COMFY_REF)


In [ ]:
import hashlib
import json
import time
import urllib.request
from pathlib import Path
DOWNLOADS = json.loads(r"""
[
  {
    "filename": "qwen2.5-3b-instruct-q4_k_m.gguf",
    "url": "https://huggingface.co/Qwen/Qwen2.5-3B-Instruct-GGUF/resolve/main/qwen2.5-3b-instruct-q4_k_m.gguf?download=true",
    "sha256": "626b4a6678b86442240e33df819e00132d3ba7dddfe1cdc4fbb18e0a9615c62d",
    "bytes": 2104932768,
    "dest": "/content/aidungeon-local/models/llm/qwen2.5-3b-instruct-q4_k_m.gguf",
    "auth_secret": "",
    "role": "weights",
    "group": "llm"
  },
  {
    "filename": "flux-2-klein-4b-fp8.safetensors",
    "url": "https://huggingface.co/black-forest-labs/FLUX.2-klein-4B/resolve/main/flux-2-klein-4b-fp8.safetensors?download=true",
    "sha256": "97ed34fe0567e436200f2faee3939b88f2b5d99f8af2a4dc16532c4245c0ccb6",
    "bytes": null,
    "dest": "/content/aidungeon-local/ComfyUI/models/diffusion_models/flux-2-klein-4b-fp8.safetensors",
    "auth_secret": "",
    "role": "unet",
    "group": "image"
  },
  {
    "filename": "qwen_3_4b.safetensors",
    "url": "https://huggingface.co/Comfy-Org/flux2-klein-4B/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors?download=true",
    "sha256": "6c671498573ac2f7a5501502ccce8d2b08ea6ca2f661c458e708f36b36edfc5a",
    "bytes": null,
    "dest": "/content/aidungeon-local/ComfyUI/models/text_encoders/qwen_3_4b.safetensors",
    "auth_secret": "",
    "role": "clip",
    "group": "image"
  },
  {
    "filename": "flux2-vae.safetensors",
    "url": "https://huggingface.co/Comfy-Org/flux2-klein-4B/resolve/main/split_files/vae/flux2-vae.safetensors?download=true",
    "sha256": "868fe7b343cc8f3a19dbcfcafbc3d5f888802be3f89bd81b65b3621a066ce8f3",
    "bytes": null,
    "dest": "/content/aidungeon-local/ComfyUI/models/vae/flux2-vae.safetensors",
    "auth_secret": "",
    "role": "vae",
    "group": "image"
  }
]
""")
def _file_sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            chunk = f.read(1024 * 1024)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()
def fetch_one(spec):
    dest = Path(spec["dest"])
    dest.parent.mkdir(parents=True, exist_ok=True)
    expected_sha = (spec.get("sha256") or "").strip().lower()
    expected = spec.get("bytes")
    if dest.is_file() and dest.stat().st_size > 0:
        if (not expected or dest.stat().st_size == int(expected)) and (not expected_sha or _file_sha256(dest) == expected_sha):
            print("já baixado", spec["filename"], flush=True)
            return
    part = dest.with_suffix(dest.suffix + ".part")
    last = None
    for attempt in range(3):
        try:
            headers = {"User-Agent": "aidungeon-colab"}
            secret_name = spec.get("auth_secret") or ""
            if secret_name:
                key = globals().get(secret_name) or ""
                if key:
                    headers["Authorization"] = "Bearer " + key
            request = urllib.request.Request(spec["url"], headers=headers)
            hasher = hashlib.sha256()
            with urllib.request.urlopen(request, timeout=180) as resp, part.open("wb") as out:
                while True:
                    chunk = resp.read(1024 * 1024)
                    if not chunk:
                        break
                    out.write(chunk)
                    hasher.update(chunk)
            digest = hasher.hexdigest()
            if expected_sha and digest != expected_sha:
                raise ValueError("SHA256 divergente em " + spec["filename"])
            if expected and part.stat().st_size != int(expected):
                raise ValueError("tamanho inesperado em " + spec["filename"])
            part.replace(dest)
            print("ok", spec["filename"], flush=True)
            return
        except Exception as exc:
            last = exc
            time.sleep(2 * (attempt + 1))
    raise SystemExit(str(last))
for spec in DOWNLOADS:
    fetch_one(spec)


In [ ]:
import os
import subprocess
from pathlib import Path
REPO = Path('/content/aidungeon-local')
LLAMA_GIT = 'https://github.com/ggml-org/llama.cpp.git'
LLAMA_REF = '1648d6411'
src = REPO / "colab" / "llama.cpp"
binary = src / "build" / "bin" / "llama-server"
if not binary.is_file():
    if not (src / ".git").is_dir():
        src.parent.mkdir(parents=True, exist_ok=True)
        subprocess.check_call(["git", "clone", LLAMA_GIT, str(src)])
    checkout = subprocess.run(["git", "checkout", LLAMA_REF], cwd=src)
    if checkout.returncode != 0:
        subprocess.check_call(["git", "fetch", "--all"], cwd=src)
        checkout = subprocess.run(["git", "checkout", LLAMA_REF], cwd=src)
    if checkout.returncode != 0:
        raise SystemExit("git checkout do llama.cpp falhou em " + LLAMA_REF)
    subprocess.check_call(["apt-get", "update"])
    subprocess.check_call(["apt-get", "install", "-y", "build-essential", "cmake", "git"])
    subprocess.check_call([
        "cmake", "-S", str(src), "-B", str(src / "build"),
        "-DGGML_CUDA=ON", "-DCMAKE_CUDA_ARCHITECTURES=native",
    ])
    subprocess.check_call(["cmake", "--build", str(src / "build"), "--target", "llama-server", "-j", str(os.cpu_count() or 2)])
if not binary.is_file():
    raise SystemExit("llama-server não foi gerado")
weights = REPO / "models" / "llm" / 'qwen2.5-3b-instruct-q4_k_m.gguf'
if not weights.is_file():
    raise SystemExit("peso do narrador ausente: " + weights.name)
log = (REPO / "colab")
log.mkdir(parents=True, exist_ok=True)
handle = (log / "llama-server.log").open("ab")
subprocess.run(["pkill", "-f", "llama-server.*--port 1234"], check=False)
subprocess.Popen([
    str(binary), "--host", "127.0.0.1", "--port", "1234",
    "-m", str(weights), "-ngl", "99", "-c", "8192", "-np", "1",
    "-t", "4", "-b", "512", "-ub", "512",
], stdout=handle, stderr=handle, start_new_session=True)
print("llama-server", weights.name)


In [ ]:
import hashlib
import json
import os
import secrets
import subprocess
import sys
import time
import urllib.error
import urllib.request
from pathlib import Path
REPO = Path('/content/aidungeon-local')
password = secrets.token_urlsafe(12)
auth_secret = secrets.token_hex(32)
env_path = REPO / ".env"
env_path.write_text(
    "AUTH_USER=Caio\n"
    + "AUTH_PASSWORD_HASH=" + hashlib.sha256(password.encode("utf-8")).hexdigest() + "\n"
    + "AUTH_SECRET=" + auth_secret + "\n"
    + "LLM_API_KEY=\n",
    encoding="utf-8",
)
settings_path = REPO / "config" / "settings.json"
settings = json.loads(settings_path.read_text(encoding="utf-8"))
settings["llm"]["base_url"] = "http://127.0.0.1:1234/v1"
settings["llm"]["context_size"] = 8192
settings["llm"]["model_alias"] = "auto"
settings["context"]["context_size"] = 8192
settings["context"]["budgets"]["recent_window"] = 2048
settings["embeddings"]["enabled"] = False
settings["voice"]["enabled"] = False
PROFILE = json.loads('{"start_comfy": true, "image_engine": "flux2_klein", "image_files": {"unet": "flux-2-klein-4b-fp8.safetensors", "clip": "qwen_3_4b.safetensors", "vae": "flux2-vae.safetensors"}, "video_engine": "", "video_files": {}}')
comfy = settings["image"]["providers"]["comfyui"]
if PROFILE["start_comfy"]:
    comfy["base_url"] = "http://127.0.0.1:8188"
    engine = PROFILE["image_engine"]
    if engine:
        comfy["engine"] = engine
    block = comfy
    if engine == "flux2_klein":
        block = comfy["flux2"]
    elif engine == "z_image_turbo":
        block = comfy["zimage"]
    for role in ("unet", "clip", "vae"):
        filename = PROFILE["image_files"].get(role) or ""
        if filename:
            block[role] = filename
    if not engine and not any(PROFILE["image_files"].values()):
        settings["image"]["enabled"] = False
    if PROFILE["video_engine"]:
        comfy["video"]["engine"] = PROFILE["video_engine"]
    for role in ("unet", "clip", "vae", "checkpoint"):
        filename = PROFILE["video_files"].get(role) or ""
        if filename:
            comfy["video"][role] = filename
else:
    settings["image"]["enabled"] = False
settings_path.write_text(json.dumps(settings, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
def wait_http(url):
    last = None
    for _ in range(60):
        try:
            with urllib.request.urlopen(url, timeout=5) as resp:
                if resp.status < 500:
                    return
                last = resp.status
        except urllib.error.HTTPError as exc:
            if exc.code < 500:
                return
            last = exc.code
        except Exception as exc:
            last = exc.__class__.__name__
        time.sleep(5)
    raise SystemExit("health check falhou: " + url + " (" + str(last) + ")")
print("login Caio")
print("senha", password)
wait_http("http://127.0.0.1:1234/health")
if PROFILE["start_comfy"]:
    wait_http("http://127.0.0.1:8188/system_stats")
app_env = os.environ.copy()
app_env["AIDUNGEON_ENV_PATH"] = str(env_path)
(REPO / "colab").mkdir(parents=True, exist_ok=True)
log = (REPO / "colab" / "app.log").open("ab")
subprocess.run(["pkill", "-f", "uvicorn.*app.main:app"], check=False)
subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd=REPO / "backend", env=app_env, stdout=log, stderr=log, start_new_session=True,
)
wait_http("http://127.0.0.1:8000/api/health/llm")
print("jogo em http://127.0.0.1:8000")


In [ ]:
import base64
import hashlib
import hmac
import pathlib
import re
import subprocess
import time
import urllib.request
binary = pathlib.Path("/tmp/cloudflared")
if not binary.is_file():
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        binary,
    )
    binary.chmod(0o755)
subprocess.run(["pkill", "-f", "cloudflared.*127.0.0.1:8000"], check=False)
log_path = pathlib.Path("/tmp/cloudflared.log")
log = log_path.open("wb")
subprocess.Popen(
    [str(binary), "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=log, stderr=subprocess.STDOUT, start_new_session=True,
)
url = ""
for _ in range(40):
    text = log_path.read_text(encoding="utf-8", errors="replace") if log_path.exists() else ""
    found = re.search(r"https://[-0-9a-z]+\.trycloudflare\.com", text)
    if found:
        url = found.group(0)
        break
    time.sleep(1)
if not url:
    raise SystemExit("o túnel não publicou uma URL")
env_file = pathlib.Path("/content/aidungeon-local/.env")
secret_val = ""
if env_file.is_file():
    for line in env_file.read_text(encoding="utf-8").splitlines():
        if line.startswith("AUTH_SECRET="):
            secret_val = line.split("=", 1)[1].strip()
if secret_val:
    exp = int(time.time()) + 168 * 3600
    u_b64 = base64.urlsafe_b64encode(b"Caio").rstrip(b"=").decode("ascii")
    e_b64 = base64.urlsafe_b64encode(str(exp).encode("ascii")).rstrip(b"=").decode("ascii")
    sig = hmac.new(secret_val.encode("utf-8"), f"Caio.{exp}".encode("utf-8"), hashlib.sha256).hexdigest()
    print("abra (1-clique já logado):", f"{url}/?aidl_token={u_b64}.{e_b64}.{sig}")
print("abra", url)
